In [0]:
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient


from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split


In [0]:


nombre_experimento = (
    "/Users/cristopheranbus@gmail.com/"
    "iris_mlflow/experimentos"
)

artifact_location = (
    "dbfs:/Volumes/workspace/my_data/"
    "my_volumen/mlflow/iris"
)

client = MlflowClient()

experimento_existente = client.get_experiment_by_name(
    nombre_experimento
)

if experimento_existente is None:
    experiment_id = client.create_experiment(
        name=nombre_experimento,
        artifact_location=artifact_location
    )
    print("Experimento creado:", experiment_id)
else:
    experiment_id = experimento_existente.experiment_id
    print("El experimento ya existe:", experiment_id)

mlflow.set_experiment(nombre_experimento)

In [0]:
df=pd.read_csv("/Volumes/workspace/my_data/my_volumen/Iris.csv")
display(df)

y = df[["Species"]]
X = df.drop(["Id","Species"], axis=1)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)



In [0]:
import logging
import numpy as np
import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score


# Reducir mensajes internos no críticos de MLflow
logging.getLogger(
    "mlflow.tracking.context.registry"
).setLevel(logging.ERROR)


# Asegurar que y tenga forma (n_filas,)
y_train_modelo = np.asarray(y_train).ravel()
y_test_modelo = np.asarray(y_test).ravel()


with mlflow.start_run(
    run_name="random-forest-basico"
) as run:

    parametros = {
        "n_estimators": 100,
        "max_depth": 4,
        "random_state": 42
    }

    modelo = RandomForestClassifier(
        **parametros
    )

    modelo.fit(
        X_train,
        y_train_modelo
    )

    predicciones = modelo.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test_modelo,
        predicciones
    )

    f1 = f1_score(
        y_test_modelo,
        predicciones,
        average="weighted"
    )

    # Registrar parámetros
    mlflow.log_params(
        parametros
    )

    # Registrar métricas
    mlflow.log_metrics({
        "test_accuracy": accuracy,
        "test_f1_weighted": f1
    })

    # Crear firma del modelo
    signature = infer_signature(
        X_train,
        modelo.predict(X_train)
    )

    # Registrar modelo
    mlflow.sklearn.log_model(
        sk_model=modelo,
        artifact_path="modelo",
        signature=signature,
        input_example=X_train.head(5)
    )

    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 weighted: {f1:.4f}")
    print(f"Run ID: {run.info.run_id}")
    print(
        f"Artifact URI: {mlflow.get_artifact_uri()}"
    )